# Bounded negotiation prompt optimization

Case-0003 | First executable prototype | [English case](README.md) | [Caso en español](README.es.md)

Optimize the policy prompt for the next response of a synthetic negotiator. The runtime,
history handling, objective, model, evaluator, and optimizer procedure remain fixed.
This is automatic prompt optimization, not recursive optimizer improvement or model training.

**Default execution is offline.** Run all cells to inspect inputs and evaluator checks without
an API key. Live execution requires an explicit switch, model, current prices, and budget.
No live results are bundled. Python 3.10+ and a Jupyter-compatible frontend are sufficient;
the helper uses only the standard library. Open this notebook from the repository or case folder.

## 1. Objective and components

Objective → one-shot prompt → responses → fixed evaluator → failure feedback → candidates → selection.

The objective is a task specification loaded explicitly, not an automatically discovered skill.
The one-shot prompt receives the same objective and development examples as later optimization.
Only the next response to each fixed history is measured; no customer simulator is implemented.

In [ ]:
from pathlib import Path
import importlib.util
import json
import sys
from IPython.display import display, Markdown, HTML

relative = Path("research/proposals/case-0003-debt-negotiation")
roots = [Path.cwd(), *Path.cwd().parents]
CASE_DIR = next((base / relative for base in roots if (base / relative / "prompt_search.py").exists()), None)
if CASE_DIR is None:
    raise FileNotFoundError("Open the notebook inside the thinking-harness checkout")
spec = importlib.util.spec_from_file_location("negotiation_prompt_search", CASE_DIR / "prompt_search.py")
search = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = search
spec.loader.exec_module(search)
cases = search.load_cases()
display(Markdown((CASE_DIR / "objective.md").read_text()))

## 2. Inputs and evaluation

Each example contains a visible history and evaluator-only labels. Capacity is annotated from
that visible history; it is not hidden customer knowledge. The negotiator sees debts, permitted
payment counts, and history. It never receives expected actions or reference replies.

The 18 fixtures cover unknown capacity, feasible offers, limited capacity, no feasible plan,
explicit refusal, and consent. Six examples belong to each fixed partition. The same intent
families appear in all partitions, so a final score is a smoke-test result, not generalization evidence.

Rule checks verify structured plans and consent. A fixed LLM judge checks meaning, relevance,
and clarity. Its rubric needs human calibration before scientific claims. Exact wording is not
required, but the next-turn action and plan are specified by this narrow protocol.

In [ ]:
example = next(c for c in cases if c["id"] == "development-limited")
print("Negotiator input:")
print(json.dumps(search.visible(example), indent=2))
print("Evaluator-only target:")
print(json.dumps(example["expected"], indent=2))

valid = {**example["expected"], "message": "We can offer two monthly payments of 150 for debt B. Would you accept?"}
invalid = {"action": "offer", "debt": "C", "payments": 2, "installment": 300,
           "message": "Two payments of 300 are available for debt C."}
print("Valid structured plan:", search.rule_check(example, valid))
print("Infeasible structured plan:", search.rule_check(example, invalid))
print("Text quality has NOT been evaluated by these offline checks.")

## 3. Configuration and budget

Set one explicit model supporting Responses structured outputs. The same model is used in three
separate roles. It stays fixed within a run. Compare different models in separate runs.
Copy current per-million input/output USD prices from the official pricing page; cached input is
conservatively charged at the configured full rate. Prices are recorded assumptions, not live billing.

Every request reserves input bytes plus framing allowance and the maximum output tokens before
sending. Actual returned usage replaces the reservation. This is an application estimate, not a
provider-enforced dollar cap. An uncertain request stops execution without retries and retains its
reservation. All three roles count toward the same budget. Final testing also consumes this budget;
if it cannot finish, the report explicitly remains incomplete.

Store `OPENAI_API_KEY` in the process environment before launching Jupyter. Never paste it into a cell.
The default switch stays false even if the key exists. The example budget is editable and not an
execution authorization. Re-running the live cell creates and charges a new run.

In [ ]:
RUN_LIVE = False
config = search.Config(
    model="",                         # Set an explicit model identifier.
    input_usd_per_million=0.0,         # Set the current published input price.
    output_usd_per_million=0.0,        # Set the current published output price.
    max_cost_usd=1.0,
    max_total_tokens=100_000,
    max_iterations=3,
    candidates_per_iteration=2,
    max_output_tokens=2048,
    target_score=1.0,
    patience=2,
    min_gain=0.01,
    evaluation_repeats=2,
    allow_live=RUN_LIVE,
)
# Upper bound on API calls excluding duplicates and early stops:
# baseline generation + baseline validation + rounds*(development + K*(proposal + validation)) + final A/B.
n = {split: sum(c["split"] == split for c in cases) for split in ("development", "validation", "test")}
r, k, rounds = config.evaluation_repeats, config.candidates_per_iteration, config.max_iterations
max_calls = 1 + 2*n["validation"]*r + rounds*(2*n["development"]*r + k*(1+2*n["validation"]*r)) + 4*n["test"]*r
print(f"Planned upper bound: {max_calls} API calls; budget guards may stop earlier.")

## 4. Search and selection

Generate P0 once from the objective and development examples. Evaluate it on validation.
If it already meets the target without violations, skip optimization and proceed to final testing.
Otherwise, use development failures to propose K candidates per round. Evaluate each on the same
validation histories and repetitions, retaining the incumbent unless an admissible candidate
improves the success rate by at least `min_gain`. Validation details are never sent to the optimizer.

Success requires the expected structured decision, no rule or semantic violation, and a positive
relevance/clarity verdict. Report violations separately; quality cannot offset them. Repetitions
expose some model variability but do not establish statistical significance.

Stop at target, iteration limit, patience, budget, interruption, or error. Freeze the chosen prompt
before comparing it with P0 on the reserved test partition. Test results never feed optimization.
Do not tune on that partition after inspecting it and still call it untouched.

In [ ]:
report = None
if RUN_LIVE:
    report = search.run_search(config)
    print("Stop reason:", report["stop_reason"])
    print("Final comparison complete:", report["final_complete"])
    print("Evidence:", report["evidence_dir"])
    if report.get("error_type"):
        print("Execution error:", report.get("error_message", report["error_type"]))
else:
    print("Offline preview complete. No API calls were made.")

## 5. Results and evidence

A run records prompts, rejected candidates, per-example replies, judge evidence, usage, model
responses, source hashes, and the Git commit in the ignored local `runs/` directory. Nothing is
executed from generated text: the optimizer can only return a prompt. Interrupted attempts remain
visible. Review traces before sharing them and keep notebook outputs cleared in Git.

The plot shows validation scores for evaluated candidates, including rejected ones. The cost
column shows cumulative estimated expenditure. A rising selected score on reused validation data
is selection behavior; only the reserved comparison can test whether it transfers within these fixtures.

In [ ]:
import html

def render_results(result):
    points = []
    if "initial_validation" in result:
        points.append(("P0", result["initial_validation"]["score"], None, "baseline"))
    for event in result["events"]:
        if event.get("status") == "evaluated":
            points.append((f"{event['iteration']}.{event['candidate']+1}", event["validation"]["score"],
                           event.get("estimated_cost_usd"), "selected" if event["selected"] else "rejected"))
    if points:
        width, height = 620, 190
        coords = [(40 + i*540/max(1,len(points)-1), 150-110*score) for i,(_,score,_,_) in enumerate(points)]
        svg = '<svg xmlns="http://www.w3.org/2000/svg" width="620" height="190" role="img" aria-label="Validation success rate by candidate">'
        svg += '<rect width="620" height="190" fill="white"/><path d="M40 30 V150 H600" fill="none" stroke="#777"/>'
        svg += '<text x="6" y="42" font-size="12">1.0</text><text x="6" y="150" font-size="12">0.0</text>'
        for (label,score,_,status),(x,y) in zip(points,coords):
            color = '#245d83' if status != 'rejected' else '#999'
            svg += f'<circle cx="{x}" cy="{y}" r="5" fill="{color}"/><text x="{x}" y="175" text-anchor="middle" font-size="12">{html.escape(label)}</text>'
        display(HTML(svg+'</svg>'))
        table = '| Candidate | Validation success | Cumulative USD estimate | Status |\n|---|---:|---:|---|\n'
        for label,score,cost,status in points:
            amount = 'See call ledger' if cost is None else f'{cost:.6f}'
            table += f'| {label} | {score:.3f} | {amount} | {status} |\n'
        display(Markdown(table))
    print("Stop:", result["stop_reason"], "| Usage:", result.get("usage", {}))
    if result.get("initial_prompt"):
        print("INITIAL PROMPT\n", result["initial_prompt"])
        print("SELECTED PROMPT\n", result["selected_prompt"])
    if result["final_complete"]:
        print("Final initial score:", result["final_initial"]["score"])
        print("Final selected score:", result["final_selected"]["score"])
        for before,after in zip(result["final_initial"]["rows"], result["final_selected"]["rows"]):
            print(before["id"], "repeat", before["repeat"])
            print("Before:", before["reply"]["message"])
            print("After: ", after["reply"]["message"])
    else:
        print("No complete final comparison. Do not claim improvement.")

if report is not None:
    render_results(report)

## 6. Interpretation and next review

This version changes the policy prompt only. It does not train a model, change tools or orchestration,
run a customer simulator, or improve its own optimizer. An easy set may be solved immediately.
The next review should assess fixture coverage, judge reliability, failure patterns, and cost before
expanding scope. Human review and PR integration remain separate from search success.

References:
- [Case and protocol](README.md)
- [OpenAI conversation state](https://developers.openai.com/api/docs/guides/conversation-state)
- [OpenAI structured outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [OpenAI pricing](https://developers.openai.com/api/docs/pricing)
- [Automatic Prompt Optimization](https://arxiv.org/abs/2305.03495)
- [STOP](https://arxiv.org/abs/2310.02304)